> **Provenance des données** — Les fichiers produits par `utils_louisfarm.py` sont synthétiques et reproductibles. Les noms de pays contextualisent les exercices ; les observations ne proviennent pas d’une enquête ni d’une institution financière réelle. Les résultats ne décrivent pas les populations de ces pays.
> Pour votre projet, documentez la source, la date, les unités et les droits d’utilisation. Les données personnelles doivent être anonymisées.


# LouisFarm — Semaine 9 : Capstone & Professional Practice

**Objectif :** Mobiliser toutes les competences acquises pour un projet professionnel complet.

Choisissez votre track et adaptez ce template.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns; sns.set_theme(style="whitegrid")
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from scipy.stats import chi2_contingency, chi2
import warnings; warnings.filterwarnings("ignore")
import sys; sys.path.insert(0, ".")
from utils_louisfarm import gen_immobilier_abidjan, gen_microcredit_ghana, gen_cacao_ci_timeseries

print("LouisFarm Data Analytics Academy")
print("SEMAINE 9 — CAPSTONE & PROFESSIONAL PRACTICE")
print("=" * 55)
print("Selectionnez votre track (A/B/C) et adaptez ce template.")
print()
print("Ce notebook couvre :")
print("  - Clustering (K-Means + PCA)")
print("  - A/B Testing (test du chi-carre)")
print("  - Synthese des competences acquises")

## Competence S9.1 — Clustering K-Means

In [ ]:
# CLUSTERING K-MEANS (competence S9)
print("COMPETENCE S9 — Clustering : Segmentation clients immobilier Abidjan")
print("=" * 65)

df = gen_immobilier_abidjan(n=2000)

# Features pour le clustering
features = ["surface_m2","nbr_chambres","distance_centre_km","prix_fcfa","annee_construction"]
X_clust = df[features].dropna()

# Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clust)

# Choisir k avec silhouette score
silhouette_scores = {}
inertias = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    silhouette_scores[k] = silhouette_score(X_scaled, labels)
    inertias[k] = km.inertia_

print("Silhouette scores par k :")
best_k = max(silhouette_scores, key=silhouette_scores.get)
for k, sc in silhouette_scores.items():
    marker = " <-- MEILLEUR" if k==best_k else ""
    print(f"  k={k}: {sc:.4f}{marker}")

# Fit du meilleur modele
km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["segment"] = km_best.fit_predict(X_scaled)

print(f"\nSegmentation retenue : {best_k} clusters")
print("\nPROFIL DE CHAQUE SEGMENT :")
profil = df.groupby("segment")[features].mean().round(1)
print(profil)

## Competence S9.2 — Visualisation PCA

In [ ]:
# VISUALISATION DES CLUSTERS AVEC PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df["pca1"] = X_pca[:,0]
df["pca2"] = X_pca[:,1]

explained = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Semaine 9 — Segmentation immobilier Abidjan (K-Means + PCA)", fontweight="bold")

colors = ["#2E86AB","#C73E1D","#F18F01","#3B1F2B"]
for seg in range(best_k):
    mask = df["segment"] == seg
    sub = df[mask]
    n_seg = mask.sum()
    avg_prix = sub["prix_fcfa"].mean()/1e6
    axes[0].scatter(sub["pca1"], sub["pca2"], alpha=0.4, s=15, color=colors[seg],
                    label=f"Seg {seg} (n={n_seg}, moy={avg_prix:.0f}M)")

axes[0].set_title(f"Clusters dans lespace PCA\n(PC1={explained[0]*100:.0f}%, PC2={explained[1]*100:.0f}%)")
axes[0].set_xlabel("Composante principale 1")
axes[0].set_ylabel("Composante principale 2")
axes[0].legend(fontsize=8)

# Elbow curve
axes[1].plot(list(inertias.keys()), list(inertias.values()), "o-", color="#2E86AB", linewidth=2)
axes[1].axvline(best_k, color="#C73E1D", linestyle="--", label=f"k optimal={best_k}")
axes[1].set_xlabel("Nombre de clusters k")
axes[1].set_ylabel("Inertie (within-cluster)")
axes[1].set_title("Methode du coude")
axes[1].legend()

plt.tight_layout()
plt.savefig("./s9_clustering.png", dpi=100, bbox_inches="tight")
plt.show()

## Competence S9.3 — A/B Testing

In [ ]:
# A/B TESTING (Chi-square) 
print("COMPETENCE S9 — A/B Testing : Test d efficacite d une campagne mobile money")
print("=" * 70)

np.random.seed(42)
n_controle = 1500
n_test_grp = 1500

# Groupe controle (pas de promotion)
conversions_ctrl = np.random.binomial(1, 0.08, n_controle)
# Groupe test (promotion SMS)
conversions_test = np.random.binomial(1, 0.11, n_test_grp)

print(f"Groupe controle (sans promo) : {n_controle} utilisateurs")
print(f"  Conversions : {conversions_ctrl.sum()} ({conversions_ctrl.mean()*100:.1f}%)")
print(f"Groupe test (avec promo SMS) : {n_test_grp} utilisateurs")
print(f"  Conversions : {conversions_test.sum()} ({conversions_test.mean()*100:.1f}%)")

# Test du chi-carre
table = np.array([
    [conversions_ctrl.sum(), n_controle-conversions_ctrl.sum()],
    [conversions_test.sum(), n_test_grp-conversions_test.sum()]
])
chi2_stat, p_value, dof, expected = chi2_contingency(table)
alpha = 0.05

print(f"\nTEST DU CHI-CARRE :")
print(f"  Chi2 = {chi2_stat:.4f} | p-value = {p_value:.4f} | ddl = {dof}")
print(f"  Seuil alpha = {alpha}")

if p_value < alpha:
    lift = (conversions_test.mean() - conversions_ctrl.mean()) / conversions_ctrl.mean() * 100
    print(f"\n  CONCLUSION : Difference STATISTIQUEMENT SIGNIFICATIVE (p={p_value:.4f} < {alpha})")
    print(f"  La promotion SMS augmente les conversions de {lift:.0f}%")
    print(f"  Recommandation : DEPLOYER la campagne SMS a grande echelle")
else:
    print(f"\n  CONCLUSION : Pas de difference significative (p={p_value:.4f} >= {alpha})")
    print(f"  Recommandation : NE PAS deployer, tester une autre strategie")

## Bilan Portfolio

In [ ]:
# BILAN DU PARCOURS — PORTFOLIO
print("=" * 60)
print("BILAN — LouisFarm Data Analytics Academy")
print("9 Semaines | Parcours Applied Track")
print("=" * 60)

portfolio = {
    "S1 - Foundations": {
        "Livrable": "Diagnostic dataset Coopérative Togo",
        "Technos": "Python, NumPy, Pandas",
        "Statut": "✅"
    },
    "S2 - Wrangling": {
        "Livrable": "Dataset propre + Data Quality Report",
        "Technos": "Pandas avancé, .merge(), .groupby()",
        "Statut": "✅"
    },
    "S3 - EDA Stats": {
        "Livrable": "Analyse Agriculture Togo + Synthèse",
        "Technos": "SciPy, Corrélation, Tests Mann-Whitney",
        "Statut": "✅"
    },
    "S4 - Visualization": {
        "Livrable": "Analytical Storyboard Inclusion Financière CI",
        "Technos": "Matplotlib, Seaborn, Plotly",
        "Statut": "✅"
    },
    "S5 - SQL": {
        "Livrable": "Rapport SQL MFB Bénin (CTE + Window Functions)",
        "Technos": "SQLite, pandas.read_sql, pymongo",
        "Statut": "✅"
    },
    "S6 - Régression": {
        "Livrable": "Modèle prix immobilier Abidjan (Ridge pipeline)",
        "Technos": "sklearn, Pipeline, Ridge, R2, MAE",
        "Statut": "✅"
    },
    "S7 - Classification": {
        "Livrable": "Scoring crédit Ghana (RF + GBM + AUC)",
        "Technos": "RandomForest, GradientBoosting, imblearn",
        "Statut": "✅"
    },
    "S8 - Time Series": {
        "Livrable": "Prévision prix cacao BCC CI + Note coopératives",
        "Technos": "statsmodels, ARIMA, ACF/PACF, intervalles de confiance",
        "Statut": "✅"
    },
    "S9 - Capstone": {
        "Livrable": "Projet professionnel complet (Track A/B/C)",
        "Technos": "K-Means, PCA, Chi-square, Storytelling complet",
        "Statut": "🔄 EN COURS"
    },
}

for semaine, details in portfolio.items():
    print(f"\n  {details['Statut']} {semaine}")
    print(f"     Livrable : {details['Livrable']}")
    print(f"     Technos  : {details['Technos']}")

print()
print("CERTIFICATION CONDITIONNEE A :")
print("  ✅ 100% modules completes")
print("  ✅ CRT >= 70% toutes semaines")
print("  🎯 Capstone >= 75%")
print()
print("  LouisFarm Data Analytics Professional — 9W Applied Track")